In [ ]:
import numpy as np
import pandas as pd

import torch
from torch.utils.data import DataLoader, Dataset, Subset
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
from torchvision import transforms
from pathlib import Path
from PIL import Image
import time
import torchvision.datasets as dataset
from torch.optim.lr_scheduler import ReduceLROnPlateau
import matplotlib.pyplot as plt

import skimage
from skimage.transform import resize
from sklearn.model_selection import StratifiedShuffleSplit


import random

import os

# Verifica disponibilità della GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
class EarlyStopping:
    def __init__(self, patience=7, verbose=True, delta=0, path='best_model.pth'):
        self.patience = patience
        self.verbose = verbose
        self.delta = delta
        self.path = path
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_loss_min = float('inf')

    def __call__(self, val_loss, model, optimizer):
        score = val_loss

        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model, optimizer)
        elif score > self.best_score - self.delta:
            self.counter += 1
            if self.verbose:
                print(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model, optimizer)
            self.counter = 0

    def save_checkpoint(self, val_loss, model, optimizer):
        if self.verbose:
            print(f'Validation loss decreased ({self.val_loss_min:.6f} --> {val_loss:.6f}).  Saving model and optimizer ...')
        torch.save({
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
        }, self.path)
        self.val_loss_min = val_loss



In [ ]:
def train_model(model, optimizer, loss_class, num_epochs):
    history_loss = []
    history_loss_val = []
    history_acc = []
    history_acc_val = []
    early_stopping = EarlyStopping(patience=7, verbose=True)
    model_weights_path = "weights/best_model.pth"
    
    if os.path.exists(model_weights_path):
        checkpoint = torch.load(model_weights_path, map_location=torch.device('cpu'))
        if 'model_state_dict' in checkpoint and 'optimizer_state_dict' in checkpoint:
            model.load_state_dict(checkpoint['model_state_dict'])
            optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
            print("Model and optimizer states loaded successfully.")
        else:
            model.load_state_dict(checkpoint)
            print("Model state loaded successfully.")
    else:
        print("Model weights file not found.")
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    
   

    for epoch in range(num_epochs):
        # Training phase
        model.train()
        train_loss = 0.0
        correct_t, total_t = 0, 0

        for batch in train_loader:
            data, labels = batch
            data, labels = data.to(device), labels.to(device)
            #print(f"Model is on device: {next(model.parameters()).device}")
            
            optimizer.zero_grad()
            class_output = model(data)
            #print(f"Output shape: {class_output.shape}") 
            loss = loss_class(class_output, labels)
            train_loss += loss.item()
            loss.backward()
            optimizer.step()
            
            preds = torch.argmax(class_output, 1)
            correct_t += (preds == labels).sum().item()
            total_t += labels.size(0)

        train_loss_mean = train_loss / len(train_loader)
        acc_t = correct_t / total_t

        history_loss.append(train_loss_mean)
        history_acc.append(acc_t)
        
        print(f"Epoch {epoch + 1}: Train Loss = {train_loss_mean:.4f}")
        print(f"Train Accuracy = {acc_t * 100:.2f}%")

        # Validation phase
        model.eval()
        val_loss = 0.0
        correct, total = 0, 0
        
        with torch.no_grad():
            for batch in val_loader:
                data, labels = batch
                data, labels = data.to(device), labels.to(device)
                class_output = model(data)
                loss = loss_class(class_output, labels)
                val_loss += loss.item()
                
                preds = torch.argmax(class_output, 1)
                correct += (preds == labels).sum().item()
                total += labels.size(0)
               
        val_loss_mean = val_loss / len(val_loader)
        acc = correct / total

        history_loss_val.append(val_loss_mean)
        history_acc_val.append(acc)
        
        print(f"Epoch {epoch + 1}: Validation Loss = {val_loss_mean:.4f}")
        print(f"Validation Accuracy = {acc * 100:.2f}%")
        
        scheduler.step(val_loss_mean)
        early_stopping(val_loss_mean, model, optimizer)
        if early_stopping.early_stop:
            print("Early stopping triggered")
            break
            
        # Print learning rate for the current epoch
        current_lr = optimizer.param_groups[0]['lr']
        print(f"Current learning rate: {current_lr}")
        
        current_time = time.localtime()
        formatted_time = time.strftime("%H:%M:%S", current_time)
        print("Current time:", formatted_time, "\n")

    #history_loss_val.pop(0)
    #history_loss.pop(0)
    # Plotting the training and validation loss and accuracy
    plt.figure(figsize=(14, 6))
    
    plt.subplot(1, 2, 1)
    plt.title("Loss Function")
    plt.plot(history_loss, label="Train Loss")
    plt.plot(history_loss_val, label="Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    
    plt.subplot(1, 2, 2)
    plt.title("Accuracy")
    plt.plot(history_acc, label="Train Accuracy")
    plt.plot(history_acc_val, label="Validation Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    
    plt.show()

In [ ]:
def test_model(model, data_loader):
    model_weights_path = "best_model.pth"
    if os.path.exists(model_weights_path):
        checkpoint = torch.load(model_weights_path, map_location=torch.device('cpu'))
        if 'model_state_dict' in checkpoint:
            model.load_state_dict(checkpoint['model_state_dict'])
            print("Model state loaded successfully.")
        else:
            model.load_state_dict(checkpoint)
            print("Model state loaded successfully.")
    else:
        print("Model weights file not found.")
        return

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    model.eval()

    test_image_ids = []
    predicted_classes = []

    with torch.no_grad():
        for batch_idx, batch in enumerate(data_loader):
            
            batch = batch.to(device)
            
            for img_idx in range(batch.size(0)):
                img = batch[img_idx].unsqueeze(0).to(device)  # Adds a batch dimension of size 1
                class_outputs = model(img)
                probabilities = F.softmax(class_outputs, dim=1)

                # Get the predicted class by finding the index with the maximum probability
                _, predicted_class = torch.max(probabilities, dim=1)

                # Collect results for submission
                image_path = data_loader.dataset.image_paths[batch_idx * data_loader.batch_size + img_idx]
                image_id = os.path.basename(image_path)
                test_image_ids.append(image_id)
                predicted_classes.append(predicted_class.item())
                
                probability = probabilities[0, predicted_class.item()].item()
                print(f"Predicted class: {predicted_class.item()}, Probability: {probability:.4f}")

                # Visualize the first image
                if batch_idx == 0 and img_idx == 0:
                    img_np = img.squeeze(0).cpu().numpy().transpose(1, 2, 0)
                    print("Original Image:")
                    plt.imshow(img_np)
                    plt.show()
              

    # Create the submission DataFrame
    submission_df = pd.DataFrame({'image': test_image_ids, 'class': predicted_classes})
    return submission_df

                        

In [ ]:
# Define transformations
transform_train = transforms.Compose([
    transforms.Resize((256, 256)),  # Resize to 256x256 pixels
    transforms.RandomRotation(degrees=30),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

transform = transforms.Compose([
    transforms.Resize((256, 256)),  # Resize to 256x256 pixels
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Updated Dataset class
class ChallengeTrainDataset(Dataset):
    def __init__(self, dataframe, datafd, transform=None, background_colors=None):
        self.dataframe = dataframe
        self.datafd = datafd
        self.transform = transform
        self.background_colors = background_colors

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        image_path = row['image']
        image = Image.open(os.path.join(self.datafd, image_path)).convert('RGB')
        label = row['class']
        if self.transform:
            image = self.transform(image)

        return image, label

# Dataset for test
class ChallengeTestDataset(Dataset):
    def __init__(self, data_dir, transform=None):
        self.data_dir = data_dir
        self.image_paths = os.listdir(data_dir)
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = os.path.join(self.data_dir, self.image_paths[idx])
        image = Image.open(image_path).convert('RGB') 

        if self.transform:
            image = self.transform(image)
        return image

# Carica il CSV
boxe_csv = pd.read_csv('new_dataset/train.csv')

# Initialize the test dataset and loader
test_dataset = ChallengeTestDataset(
    data_dir='new_dataset/test',
    transform=transform
)

test_loader = DataLoader(test_dataset, batch_size=200, shuffle=False)

In [ ]:
# Load dataset
boxe_csv = pd.read_csv("new_dataset/train.csv")

# Ensure stratified sampling (equal distribution of classes)
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_index, val_index = next(sss.split(boxe_csv, boxe_csv['class']))

# Create training and validation Sets
train_df = boxe_csv.iloc[train_index]
val_df = boxe_csv.iloc[val_index]


train_dataset = ChallengeTrainDataset(
    dataframe=train_df,
    datafd='new_dataset/train',
    transform=transform_train
)

val_dataset = ChallengeTrainDataset(
    dataframe=val_df,
    datafd='/new_dataset/train',
    transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

print(boxe_csv['class'].unique())

# Model initialization
from torchvision.models import resnet18

net = resnet18(weights=None)
num_classes = 9
net.fc = nn.Linear(net.fc.in_features, num_classes)

net.to(torch.device('cuda' if torch.cuda.is_available() else 'cpu'))

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adagrad(net.parameters(), lr=0.001)

scheduler = ReduceLROnPlateau(optimizer, 'min', patience=3, verbose=True)
train_model(net, optimizer, criterion, num_epochs=100)

In [ ]:
print("Inizio il testing")
submission_df = test_model(net, test_loader)
submission_df = submission_df.sort_values(by='image')
submission_df.to_csv('submission.csv', index=False)
print("Submission file saved as 'submission.csv'")
